In [ ]:
# create a python environment with python 3.11, install the packages you need before doing this:
import re 
import polars as pl 

In [ ]:
# read in transcript 10
transcript10 = open("example_transcript/Session10_Transcript.txt").read() # the file-name has to be altered according to your folder structure!
# have a look at how the data is structured
transcript10
# so it seems every speaker-turn starts with a new line: \n, but there are no commas separating the different elements
# we have a pattern though, every turn starts with: Speaker:, every turn ends with a timestamp: 00:00:00.0
# so let's use some regex to sort this out

In [ ]:
''' we will now find every turn a speaker makes by splitting the text everytime 
a newline "\n" is followed by "(?=)" a word character "\w" and 0 or more word characters or spaces "[\w\s]*"
followed by a colon ":"; resulting in this regex expression r"\n(?=\w[\w\s]*:)" '''

# we will store these in a list for now.
chunks = re.split(r"\n(?=\w[\w\s]*:)", transcript10)
# have a look
chunks


In [ ]:
''' cool, now we turn this into a dataframe, but we want to have different columns for speaker, speech and timestamp
so we need to chunk our chunks with more regex: r"^(\w[\w\s]*):\s+(.+?)(?:\s+(\d{2}:\d{2}:\d{2}[\.\d]*))?$"
this finds groups "(...)" within the beginning "^" and the end "$" of a line. The first group, we know from the step before,
the second "(.+?)" captures anything that comes after a colon ":" and a space "\s" and the last captures two digits "\d{2}" that are followed by 
colon, which is followed by two digits, etc. so HH:MM:SS with the option of being followed by a dot and more digits "[\.\d]*
- you don't have to completely understand regex, and absolutely don't need to be able to write those formulars yourself. That's what ai was invented for. 
(Or stack overflow before it)
'''
# create a list to store our new chunk-chunks
rows = []
for chunk in chunks:
    match = re.match(r"^(\w[\w\s]*):\s+(.+?)(?:\s+(\d{2}:\d{2}:\d{2}[\.\d]*))?$", chunk, re.DOTALL) # find the groups
    if match:
        rows.append(match.groups())  # add the groups to the list (speaker, speech, time)

df = pl.DataFrame(rows, schema=["speaker", "speech", "time"]) # make a dataframe out of it

print(df.head(10)) # look at the first few rows
print(df.dtypes) # what datatypes do we have?
print(df.describe)
# looks good!

In [ ]:
# so let's look at our speakers:
print(df["speaker"].unique())

# shoot! seems like there's a typo 

In [ ]:
# fix that
df = df.with_columns([
    pl.col("speaker")
        .str.replace("MArk", "Mark")
        .alias("speaker")
])

print(df["speaker"].unique()) # check


In [ ]:
'''okay, now we want to structure the data a bit further, how about extracting events?
They are coded this way in the transcripts:
Reading Story - *1* blabla *1*
Reading Poem - *2* blabla *1* 
Citing Text - <blabla>
Citing Group Member - "blabla"

So let's define a function for that'''
def get_event(speech):
    if re.search(r'\*1\*', speech):
        return "Reading Story"
    if re.search(r'\*2\*', speech):
        return "Reading Poem"
    if re.search(r'<.+?>', speech, re.DOTALL):  # re.DOTALL so < > can span lines
        return "Citing Text"
    if re.search(r'".+?"', speech, re.DOTALL):  # same for quotes spanning lines
        return "Citing GM"
    return None

In [ ]:
# and apply
df = df.with_columns(
    pl.col("speech").map_elements(get_event, return_dtype=pl.String).alias("event")
)

# let's have a look
df.describe()

# we have a null-count in event! but that's fine, not every line has one, 44 lines sound short though!

In [ ]:
# look at the events we identified
with pl.Config(tbl_rows=50, fmt_str_lengths=200, tbl_width_chars=1000):
    print(df.filter(pl.col("event").is_not_null()))

# well, the function works in identifying the events, but we run into problems, when the event goes over several lines with no event marker


In [ ]:
# so now we need to redefine our function to not only find events but also check whether they are completed

def get_event(speech):
    if re.search(r'\*1\*', speech):
        # even number of *1* markers = self-contained, odd = still open
        complete = len(re.findall(r'\*1\*', speech)) % 2 == 0
        return "Reading Story", complete
    if re.search(r'\*2\*', speech):
        complete = len(re.findall(r'\*2\*', speech)) % 2 == 0
        return "Reading Poem", complete
    if re.search(r'[<>]', speech):
        # complete if both < and > are present in the same chunk
        complete = bool(re.search(r'<', speech) and re.search(r'>', speech))
        return "Citing Text", complete
    if re.search(r'"', speech):
        # even number of quotes = self-contained
        complete = len(re.findall(r'"', speech)) % 2 == 0
        return "Citing GM", complete
    return None, False


In [ ]:
# and apply
events = []
current_event = None

for speaker, speech, time in df[["speaker", "speech", "time"]].iter_rows():
    event, is_complete = get_event(speech)

    if event is not None:
        if current_event == event:
            current_event = None
        elif not is_complete:
            current_event = event
    else:
        event = current_event

    events.append(event)

df = df.with_columns(pl.Series("event", events))

df.describe() # now we have a count of 190 instances, which sounds more realistic


In [ ]:
# looks good
with pl.Config(tbl_rows=50, fmt_str_lengths=200, tbl_width_chars=1000):
    print(df.filter(pl.col("event").is_not_null()))

In [ ]:
# Alternatively we could move this step into how we read the data, which might be more efficient if we want to write a function for the entire dataset
rows = []
current_event = None # define switch for long events
    
for chunk in chunks:
    match = re.match(r"^(\w[\w\s]*):\s+(.+?)(?:\s+(\d{2}:\d{2}:\d{2}[\.\d]*))?$", chunk, re.DOTALL)
    if match:
        speaker, speech, time = match.groups()
        event, is_complete = get_event(speech)

        if event is not None:
            if current_event == event:
                # we're already inside this block — this is the closing marker
                current_event = None
            elif not is_complete:
                # opening marker without a close — start tracking
                current_event = event
                # if is_complete, it's self-contained — just assign it, don't toggle
        else:
            # no marker — inherit ongoing event if inside a block
            event = current_event

        rows.append((speaker, speech, time, event))

df = pl.DataFrame(
    rows,
    schema=["speaker", "speech", "time", "event"],
    orient="row"
)

df.describe() 

In [ ]:
with pl.Config(tbl_rows=50, fmt_str_lengths=200, tbl_width_chars=1000):
    print(df.filter(pl.col("event").is_not_null()))

In [ ]:
# but we could also write a function to inspect the transcript for mistakes in the events 

full_text = " ".join(df["speech"].drop_nulls().to_list()) # we concatenate all of the cells to one text
issues = []
# we count how often the markers appear
counts = {
    "*1*": len(re.findall(r'\*1\*', full_text)),
    "*2*": len(re.findall(r'\*2\*', full_text)),
    '"':   len(re.findall(r'"', full_text)),
    "<":   len(re.findall(r'<', full_text)),
    ">":   len(re.findall(r'>', full_text)),
}

# check for imbalances
if counts["*1*"] % 2 != 0: 
    issues.append(f"odd number of *1* markers ({counts['*1*']})")
if counts["*2*"] % 2 != 0:
    issues.append(f"odd number of *2* markers ({counts['*2*']})")
quote = '"'
if counts['"'] % 2 != 0:
    issues.append(f"odd number of {quote} markers ({counts[quote]})")
if counts["<"] != counts[">"]:
    issues.append(f"{counts['<']} opening < but {counts['>']} closing >")
    # we check for the position of problematic "<" and ">" because they appear much more frequent and are more difficult to spot than *1* / *2* or ""
    if counts["<"] > counts[">"]: # here we check where a "<" appears without a ">" before it
        closed = True
        issue = False
        for row in df.iter_rows(named=True):
            if not row["speech"]:
                continue
            for char in row["speech"]:
                if (char == "<") and (closed == True):
                    closed = False
                elif (char == ">") and (closed == False):
                    closed = True
                elif (char == "<") and (closed == False):
                    issues.append(f"unclosed before this: '{row['speech']}'")
                                
    if counts[">"] > counts["<"]: # here we check where a ">" appears without a "<" before it
        opened = False
        for row in df.iter_rows(named=True):
            if not row["speech"]:
                continue
            for char in row["speech"]:
                if (char == "<") and (opened == False):
                    opened =True
                elif (char == ">") and (opened == True):
                    opened = False
                elif (char == "<") and (opened == True):
                    issues.append(f"unopened before this: '{row['speech']}'")




if not issues:
    print("All markers are balanced!")
else:
    print(f"Found {len(issues)} issue(s):\n")
    for issue in issues:
        print(issue)

# but here we don't seem to have problems. - mostly because I already dealt with this, but thought it would be good for you to see. 
      